# 03 — KnottedGraph vs Topoly: focused paper scaling

This notebook performs the main reusable benchmark experiment for three paper views and then adds a separate **topology-hard** crossing experiment.

1. **Crossing scaling:** runtime versus projected crossing count $c$, aggregated across a fixed panel of heterogeneous connected trivalent graphs.
2. **Vertex scaling:** runtime versus $V$, pooling rows with the same $V$ across the crossing grid.
3. **Edge scaling:** runtime versus $E$, analogously pooling rows with the same $E$.
4. **Essential-crossing scaling:** runtime versus a certified minimal crossing number using theta graphs with a $T(2,n)$ constituent cycle.

For every timing comparison, graph construction and graph-to-PD conversion happen **outside** the timed region, KnottedGraph and Topoly receive the same PD code, and successful paired evaluations must agree polynomially up to the benchmark's explicitly allowed Laurent-unit/variable-orientation conventions.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, os, subprocess, sys
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph repository checkout.")
DEV = ROOT / "dev"
if str(DEV) not in sys.path:
    sys.path.insert(0, str(DEV))

import knotted_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error

print("Python executable:", sys.executable)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())
if not native_available():
    raise RuntimeError(
        "This performance benchmark requires the compiled native Yamada backend. "
        "Install the checkout with `python -m pip install -e .` in the active environment."
    )

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install benchmark dependencies with: python -m pip install -e '.[benchmark]'") from exc

OUT = ROOT / "User_guide" / "benchmarks"
RES = OUT / "results_latest"
FIG = OUT / "figures_latest"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
print("Topoly:", Path(topoly.__file__).resolve())


## 1. Configuration

The main controlled-crossing benchmark keeps the same heterogeneous trivalent graph-size panel at each crossing count. Crossing counts use the doubling grid $1,2,4,8,\ldots$. The new topology-hard section has its own odd-$n$ grid.


In [ ]:
IS_CI = os.environ.get("CI", "").lower() == "true"
PROFILE = "smoke" if IS_CI else "paper"
CROSSING_GRAPHS = 3 if IS_CI else 21
MAX_PROJECTED_CROSSINGS = 8 if IS_CI else 80
SIZE_SCALING_CROSSINGS = 8
TIMEOUT_S = 10 if IS_CI else 120
CENSOR_FRONTIER = 2

raw_csv = RES / "topoly_yamada_paper_scaling_raw.csv"
aggregate_csv = RES / "topoly_yamada_paper_scaling_aggregate.csv"

if MAX_PROJECTED_CROSSINGS < 1:
    raise ValueError("MAX_PROJECTED_CROSSINGS must be >= 1")
if SIZE_SCALING_CROSSINGS < 1:
    raise ValueError("SIZE_SCALING_CROSSINGS must be >= 1")
print(
    f"mode={PROFILE}, graph sizes/c={CROSSING_GRAPHS}, "
    f"crossing target={MAX_PROJECTED_CROSSINGS}, "
    f"timeout/framework/sample={TIMEOUT_S}s"
)


In [ ]:
def _load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not load {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def run_streamed_benchmark(cmd, *, total, description):
    print("Running:", " ".join(map(str, cmd)))
    process = subprocess.Popen(
        cmd, cwd=ROOT, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Benchmark subprocess did not expose stdout.")

    streamed_rows = []
    summary_rows = None
    bar = tqdm(total=total, desc=description, unit="sample", dynamic_ncols=True, leave=True)

    def _time_text(row, framework):
        status = row.get(f"{framework}_status", "?")
        value = row.get(f"{framework}_s")
        if status == "ok" and value is not None:
            return f"{float(value):.6f} s"
        if status == "timeout":
            return f"TIMEOUT (>={row.get('timeout_s', '?')} s)"
        if status == "error":
            return "ERROR"
        if status == "skipped_after_censor_frontier":
            return "SKIPPED"
        return status.upper()

    for raw_line in process.stdout:
        line = raw_line.rstrip()
        if not line:
            continue
        if line.startswith("SUMMARY="):
            summary_rows = json.loads(line[len("SUMMARY="):])
            continue
        if line.startswith("{"):
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                bar.write(line)
                continue
            if "family" in row:
                streamed_rows.append(row)
                kg = _time_text(row, "knottedgraph")
                tp = _time_text(row, "topoly")
                speed = ""
                if row.get("knottedgraph_status") == row.get("topoly_status") == "ok":
                    if row.get("knottedgraph_s") and row.get("topoly_s"):
                        speed = f" | Topoly/KnottedGraph={float(row['topoly_s'])/float(row['knottedgraph_s']):.2f}x"
                bar.write(
                    f"TIMING {row.get('family')} c={row.get('crossings')} V={row.get('V')} E={row.get('E')} "
                    f"sample={row.get('sample', '?')} | KnottedGraph={kg} | Topoly={tp}{speed}"
                )
                bar.update(1)
                continue
        bar.write(line)

    return_code = process.wait()
    bar.close()
    if return_code:
        raise RuntimeError(f"Benchmark failed with exit code {return_code}")
    rows = summary_rows if summary_rows is not None else streamed_rows
    if not rows:
        raise RuntimeError("Benchmark completed without sample rows.")
    print(f"completed {len(rows)} sample records")
    return rows


## 2. Run the reusable controlled-crossing ensemble

At every projected crossing count $c$, the same trivalent graph-size panel is used. Because every graph is trivalent, $E=3V/2$. These crossings control the supplied PD complexity, but they are **not claimed to be topologically unavoidable**.


In [ ]:
paper_script = ROOT / "dev" / "benchmark_topoly_paper_scaling.py"
env = dict(os.environ)
env.pop("PYTHONPATH", None)
paper_module = _load_module(paper_script, "kg_topoly_paper_scaling_notebook_plan")
plan = paper_module.paper_plan(PROFILE, CROSSING_GRAPHS, MAX_PROJECTED_CROSSINGS, SIZE_SCALING_CROSSINGS)
assert set(plan) == {"crossings_graph_ensemble"}
crossing_grid = plan["crossings_graph_ensemble"]["x_values"]
assert crossing_grid[0] == 1
assert all(b == 2 * a for a, b in zip(crossing_grid, crossing_grid[1:]))
assert crossing_grid[-1] >= MAX_PROJECTED_CROSSINGS
assert SIZE_SCALING_CROSSINGS in crossing_grid

cmd = [
    sys.executable, str(paper_script),
    "--profile", PROFILE,
    "--crossing-graphs", str(CROSSING_GRAPHS),
    "--max-crossings", str(MAX_PROJECTED_CROSSINGS),
    "--size-scaling-crossings", str(SIZE_SCALING_CROSSINGS),
    "--timeout", str(TIMEOUT_S),
    "--censor-frontier", str(CENSOR_FRONTIER),
]
rows = run_streamed_benchmark(
    cmd, total=len(crossing_grid) * CROSSING_GRAPHS, description="Paper Yamada scaling"
)


## 3. Acceptance checks and raw-data export

The checks enforce the repeated $V/E$ panel and preserve the independent KnottedGraph-versus-Topoly polynomial agreement test for every paired evaluation that completes.


In [ ]:
from collections import defaultdict

assert {row["family"] for row in rows} == {"crossings_graph_ensemble"}
groups = defaultdict(list)
for row in rows:
    groups[int(row["crossings"])].append(row)

expected_panel = None
for crossings, group in sorted(groups.items()):
    assert len(group) == CROSSING_GRAPHS
    panel = sorted((int(row["V"]), int(row["E"])) for row in group)
    assert len(set(panel)) == CROSSING_GRAPHS
    assert all(bool(row["connected"]) for row in group)
    assert all(int(row["regular_degree"]) == 3 for row in group)
    assert all(int(row["E"]) == 3 * int(row["V"]) // 2 for row in group)
    if expected_panel is None:
        expected_panel = panel
    else:
        assert panel == expected_panel

for row in rows:
    if row["correctness"] == "PASS":
        assert row["knottedgraph_status"] == row["topoly_status"] == "ok"
        assert row["pd_hash"]

keys = list(dict.fromkeys(key for row in rows for key in row))
with raw_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys)
    writer.writeheader()
    writer.writerows(rows)
print("PASS: controlled-crossing ensemble structure and paired correctness checks passed.")
print("V/E panel:", expected_panel)


## 4. Generate the three original paper views

The crossing plot aggregates at fixed supplied $c$. Vertex and edge plots pool the full repeated sample set by $V$ or $E$. Timeout observations enter those size averages at the timeout threshold, hence as conservative lower bounds.


In [ ]:
plot_script = ROOT / "dev" / "plot_topoly_paper_scaling.py"
plot_cmd = [
    sys.executable, str(plot_script), str(raw_csv),
    "--figure-dir", str(FIG),
    "--aggregate-csv", str(aggregate_csv),
]
plot_process = subprocess.run(plot_cmd, cwd=ROOT, env=env, text=True, capture_output=True)
print(plot_process.stdout)
if plot_process.returncode:
    raise RuntimeError(plot_process.stderr)
for stem in [
    "topoly_vs_knottedgraph_crossings",
    "topoly_vs_knottedgraph_vertices",
    "topoly_vs_knottedgraph_edges",
]:
    assert (FIG / f"{stem}.png").exists()
    assert (FIG / f"{stem}.pdf").exists()
assert aggregate_csv.exists()
print("PASS: original crossing, vertex, and edge figures created.")


## 5. Interpretation of the original ensemble

**Crossing view.** Each point measures performance for a supplied PD crossing count. These crossings are useful for evaluator scaling but can include Reidemeister-removable structure.

**Vertex/edge views.** Every graph is trivalent, so $E=3V/2$; the two size panels are reparameterizations of the same size dependence rather than independent evidence.


## 6. Topologically essential crossings: $T(2,n)$ constituent cycles

The controlled-crossing ensemble above measures **diagram complexity**. This section adds a distinct experiment in which the crossing count is certified by the topology of the spatial embedding.

For every odd $n\ge3$, the benchmark constructs a trivalent theta graph whose distinguished two-edge constituent cycle is the standard closure of the two-strand braid

$$
\sigma_1^n.
$$

Its closure is the torus knot $T(2,n)$ (or its mirror under orientation reversal), and

$$
\operatorname{cr}(T(2,n))=n.
$$

Hence every diagram of the whole spatial graph has at least $n$ crossings: a diagram with fewer would induce a diagram of that constituent knot with fewer than its crossing number. The generator supplies the standard $n$-crossing braid diagram and routes the third theta edge outside it, so the supplied projection attains the bound,

$$
c_{\mathrm{projected}}=c_{\min}=n.
$$

This is therefore a **topology-hard scaling test**, rather than merely a test of a Reidemeister-reducible high-crossing diagram.


In [ ]:
# Odd n values define T(2,n) constituent knots with certified crossing number n.
ESSENTIAL_TORUS_N = [3, 5] if IS_CI else [3, 5, 9, 17, 33]
essential_raw_csv = RES / "topoly_yamada_essential_torus_raw.csv"
essential_figure_png = FIG / "topoly_vs_knottedgraph_essential_torus_crossings.png"
essential_figure_pdf = FIG / "topoly_vs_knottedgraph_essential_torus_crossings.pdf"
assert ESSENTIAL_TORUS_N == sorted(set(ESSENTIAL_TORUS_N))
assert all(n >= 3 and n % 2 == 1 for n in ESSENTIAL_TORUS_N)
print("Certified T(2,n) crossing grid:", ESSENTIAL_TORUS_N)


### 6.1 Run the certified topology-hard ensemble

Here $V=2$ and $E=3$ are fixed while the embedding complexity grows. The same timeout/censoring and polynomial-agreement machinery is reused.


In [ ]:
essential_script = ROOT / "dev" / "benchmark_topoly_essential_torus_scaling.py"
essential_cmd = [
    sys.executable, str(essential_script),
    "--n-values", ",".join(map(str, ESSENTIAL_TORUS_N)),
    "--timeout", str(TIMEOUT_S),
    "--censor-frontier", str(CENSOR_FRONTIER),
]
essential_rows = run_streamed_benchmark(
    essential_cmd, total=len(ESSENTIAL_TORUS_N), description="Essential T(2,n) Yamada scaling"
)


### 6.2 Certificate and correctness checks

Three facts are kept separate: the construction certificate $T(2,n)$, PD verification that the supplied diagram has exactly $n$ crossings, and KnottedGraph-versus-Topoly Yamada agreement whenever both finish. A timeout is unevaluated, not a correctness failure.


In [ ]:
assert {row["family"] for row in essential_rows} == {"essential_torus_constituent"}
for row in essential_rows:
    n = int(row["size"])
    assert n in ESSENTIAL_TORUS_N
    assert row["constituent_knot"] == f"T(2,{n})"
    assert int(row["certified_min_crossings"]) == n
    assert int(row["crossings"]) == n
    assert bool(row["crossing_minimal_diagram"])
    assert int(row["V"]) == 2 and int(row["E"]) == 3
    assert bool(row["connected"])
    assert int(row["regular_degree"]) == 3
    if row["correctness"] == "PASS":
        assert row["knottedgraph_status"] == row["topoly_status"] == "ok"
        assert row["pd_hash"]

essential_keys = list(dict.fromkeys(key for row in essential_rows for key in row))
with essential_raw_csv.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=essential_keys)
    writer.writeheader()
    writer.writerows(essential_rows)
n_pass = sum(row["correctness"] == "PASS" for row in essential_rows)
print(f"PASS: all {len(essential_rows)} supplied T(2,n) diagrams attain the certified crossing number.")
print(f"Polynomial agreement: {n_pass} PASS, {len(essential_rows)-n_pass} unevaluated.")


### 6.3 Runtime versus certified unavoidable crossing number

Successful runtimes are plotted at their measured values. A timeout is shown at the timeout threshold and is therefore only a lower bound on the true runtime.


In [ ]:
import matplotlib.pyplot as plt

def _essential_runtime(row, framework):
    status = row[f"{framework}_status"]
    if status == "ok":
        return float(row[f"{framework}_s"]), False
    if status == "timeout":
        return float(row["timeout_s"]), True
    return None, False

fig, ax = plt.subplots(figsize=(6.4, 4.6))
for framework, label in (("knottedgraph", "KnottedGraph"), ("topoly", "Topoly")):
    xs, ys, tx, ty = [], [], [], []
    for row in essential_rows:
        value, censored = _essential_runtime(row, framework)
        if value is None:
            continue
        n = int(row["certified_min_crossings"])
        xs.append(n); ys.append(value)
        if censored:
            tx.append(n); ty.append(value)
    if xs:
        ax.plot(xs, ys, marker="o", label=label)
    if tx:
        ax.scatter(tx, ty, marker="v", label=f"{label} timeout lower bound")
ax.set_xlabel(r"Certified minimal crossing number $c_{\min}=n$ of constituent $T(2,n)$")
ax.set_ylabel("Yamada evaluation time (s)")
ax.set_yscale("log")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(essential_figure_png, dpi=220)
fig.savefig(essential_figure_pdf)
plt.show()
assert essential_figure_png.exists() and essential_figure_pdf.exists()


### 6.4 Interpretation

| Benchmark | What $c$ measures | Crossings essential? | Purpose |
| --- | --- | --- | --- |
| Controlled crossing family | supplied PD complexity | not necessarily | raw evaluator scaling |
| $T(2,n)$ constituent family | certified minimal crossing number | **yes** | topology-hard scaling |

The second family deliberately holds the abstract theta graph fixed while increasing embedding complexity. Because the certificate comes from a constituent knot with known crossing number, Reidemeister simplification cannot reduce these examples below the reported $n$ crossings.
